## Cell 1: Feedforward neural network for building energy use

**Owner: Alex Mwamsindo**

> **Version v03 — base features, raw target**
> Inputs: 175 base columns (same as v01).
> Target: raw `MFBTU` in kBtu, standardized with fit-row mean and standard deviation.
> Purpose: test whether the log1p target matters by comparing against v01
> (MAE 5.24M ± 0.18M kBtu). Only the target changes; all other settings match v01.

#### Objectives
1. Build a reproducible feedforward neural network in TensorFlow/Keras to predict
   `MFBTU`, annual major-fuels consumption in kBtu.
2. Find the input configuration that gives the best validation performance by
   comparing the base feature set (175 columns) with the engineered set (183 columns),
   while keeping all 23 source variables.
3. Compare a raw `MFBTU` target with a `log1p(MFBTU)` target, scoring both in original kBtu.
4. Tune a small, justified set of hyperparameters (layers, units, learning rate,
   batch size, dropout, L2 regularization, early stopping) and check for overfitting
   with learning curves.
5. Evaluate the selected model across building subgroups, then score it once on
   the held-out test set.

#### Motivation
Commercial real estate acquisition analysts often need an energy estimate before
they have a building's utility history. A neural network can learn nonlinear
relationships and interactions between building size, operations, equipment and
climate that a linear model cannot represent directly. We test whether that
flexibility improves on the team's linear regression baseline and how it compares
with gradient boosting, without assuming it will.

#### Data
The team's processed 2018 CBECS file (`cbecs2018_processed.csv`, 6,357 buildings):
23 source variables with missing-value flags, survey-cap flags, one-hot encoded
categories, and eight candidate engineered features. We keep the project's fixed
70/15/15 split (4,449 / 954 / 954). Scaling is fitted on training rows only.
Validation MAE in kBtu is the main selection metric, as in the gradient boosting
notebook. The test set is used once, after model selection is complete.

In [1]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 3: Imports, package versions, and reproducibility settings
import os
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from IPython.display import display

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
tf.keras.utils.set_random_seed(SEED)            # seeds Python, NumPy, and TensorFlow together
tf.config.experimental.enable_op_determinism()  # makes TF operations repeatable (TF >= 2.9)

versions = {'Python': platform.python_version(), 'NumPy': np.__version__, 'pandas': pd.__version__,
            'scikit-learn': sklearn.__version__, 'TensorFlow': tf.__version__,
            'GPUs visible': len(tf.config.list_physical_devices('GPU'))}
display(pd.Series(versions, name='Version').to_frame())

,Version
Python,3.13.18
NumPy,2.4.6
pandas,2.2.3
scikit-learn,1.6.1
TensorFlow,2.21.0
GPUs visible,0


In [3]:
# Cell 4: Load the processed data and run integrity checks
data_path = Path('/content/drive/MyDrive/Data/cbecs2018_processed.csv')
data = pd.read_csv(data_path)

ID_COLUMNS = ['PUBID', 'split', 'MFBTU']
NUMERIC = ['SQFT', 'FLCEILHT', 'WKHRS', 'NWKER', 'MONUSE', 'HDD65', 'CDD65', 'HEATP', 'COOLP', 'OCCUPYP']
ENGINEERED = ['log1p_SQFT', 'log1p_NWKER', 'workers_per_1000_sqft', 'weekly_floor_area_hours',
              'heated_floor_area', 'cooled_floor_area', 'heating_area_degree_days', 'cooling_area_degree_days']
all_features = [c for c in data.columns if c not in ID_COLUMNS]
binary_features = [c for c in all_features if c not in NUMERIC + ENGINEERED]
feature_sets = {'base': [c for c in all_features if c not in ENGINEERED], 'engineered': all_features}

# Structure and split checks
assert data.shape == (6357, 186), data.shape
assert data.PUBID.is_unique and data.PUBID.notna().all()
assert data['split'].value_counts().to_dict() == {'train': 4449, 'validation': 954, 'test': 954}
assert len(feature_sets['base']) == 175 and len(feature_sets['engineered']) == 183
# Leakage and value checks
assert not set(ID_COLUMNS) & set(all_features), 'ID, split, or target found among features'
assert np.isfinite(data[all_features + ['MFBTU']].to_numpy(dtype=float)).all()
assert data.MFBTU.gt(0).all(), 'log1p target requires positive MFBTU'
assert data[binary_features].isin([0, 1]).all().all()
assert data.SQFT.le(1_000_000).all() and data.NWKER.le(5_000).all()

# Same row order convention as the gradient boosting notebook
train = data.loc[data['split'].eq('train')].sort_values('PUBID').reset_index(drop=True)
validation = data.loc[data['split'].eq('validation')].sort_values('PUBID').reset_index(drop=True)
n_test = int(data['split'].eq('test').sum())   # test rows stay untouched until the final stage

display(pd.DataFrame({'Rows': [len(train), len(validation), n_test]},
                     index=['train', 'validation', 'test (sealed)']))
display(pd.DataFrame({'Columns': [len(NUMERIC), len(ENGINEERED), len(binary_features),
                                  len(feature_sets['base']), len(feature_sets['engineered'])]},
                     index=['Numeric source', 'Engineered', 'Binary (flags + one-hot)', 'Base total', 'Engineered total']))
print(f"Training MFBTU: median {train.MFBTU.median():,.0f} kBtu, mean {train.MFBTU.mean():,.0f} kBtu")
print(f"Training skewness: raw {train.MFBTU.skew():.2f}, log1p {np.log1p(train.MFBTU).skew():.2f}")
print('All Stage 1 integrity checks passed.')

,Rows
train,4449
validation,954
test (sealed),954


,Columns
Numeric source,10
Engineered,8
Binary (flags + one-hot),165
Base total,175
Engineered total,183


Training MFBTU: median 4,397,988 kBtu, mean 15,868,331 kBtu
Training skewness: raw 7.69, log1p -0.42
All Stage 1 integrity checks passed.


## Cell 5: Experiment configuration

Every choice that differs between versions lives in the `CONFIG` cell below:
feature set, target transform, input transformations, architecture, and training
settings. All other cells are identical across v01–v06, so differences in results
can be attributed to the configuration alone.

Early stopping monitors a fixed 15% of the training rows (seed 42), never the
official validation set. The validation set is used only to compare versions,
and the test set stays sealed. Each version is trained with three random seeds
so we can separate real improvements from run-to-run noise.

In [4]:
# Cell 6: Experiment configuration (the only code cell that changes between versions)
CONFIG = {
    'version': 'v03_raw_target',
    # Inputs
    'feature_set': 'base',            # 'base' (175 columns) or 'engineered' (183 columns)
    'log_heavy_features': False,      # v04: log1p the heavy-tailed engineered products before scaling
    'drop_sources': [],               # v05: source variables to remove (with their flags/one-hot columns)
    'drop_columns': [],               # v05: individual columns to remove (e.g., engineered features built from dropped inputs)
    # Target
    'target': 'raw',                  # 'log1p' or 'raw'; both are standardized and scored in kBtu
    # Architecture
    'hidden_units': [64, 32],
    'activation': 'relu',
    'dropout': 0.0,
    'l2': 0.0,
    # Training
    'learning_rate': 1e-3,
    'batch_size': 32,
    'max_epochs': 300,
    'patience': 20,                   # epochs without monitor improvement before stopping
    'monitor_fraction': 0.15,
    'seeds': [42, 43, 44],
}
display(pd.Series({k: str(v) for k, v in CONFIG.items()}, name='Setting').to_frame())

,Setting
version,v03_raw_target
feature_set,base
log_heavy_features,False
drop_sources,[]
drop_columns,[]
target,raw
hidden_units,"[64, 32]"
activation,relu
dropout,0.0
l2,0.0


In [5]:
# Cell 7: Preparing inputs and target for this version
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

HEAVY_TAILED = ['weekly_floor_area_hours', 'heated_floor_area', 'cooled_floor_area',
                'heating_area_degree_days', 'cooling_area_degree_days']

# 1. Columns used in this version
feature_columns = [c for c in feature_sets[CONFIG['feature_set']]
                   if not any(c == s or c.startswith(s + '__') for s in CONFIG['drop_sources'])
                   and c not in CONFIG['drop_columns']]
continuous_columns = [c for c in feature_columns if c in NUMERIC + ENGINEERED]
binary_columns = [c for c in feature_columns if c not in continuous_columns]

# 2. Fixed early-stopping rows carved from training (identical in every version)
fit_idx, monitor_idx = train_test_split(train.index, test_size=CONFIG['monitor_fraction'], random_state=SEED)
fit, monitor = train.loc[sorted(fit_idx)], train.loc[sorted(monitor_idx)]
frames = {'fit': fit, 'monitor': monitor, 'validation': validation}

# 3. Inputs: optional log of heavy tails, then standardize continuous columns on fit rows only
def transform_inputs(frame):
    values = frame[feature_columns].astype(float).copy()
    if CONFIG['log_heavy_features']:
        cols = [c for c in HEAVY_TAILED if c in values]
        values[cols] = np.log1p(values[cols])
    return values

X_frames = {name: transform_inputs(f) for name, f in frames.items()}
scaler = StandardScaler().fit(X_frames['fit'][continuous_columns])
for name in X_frames:
    X_frames[name][continuous_columns] = scaler.transform(X_frames[name][continuous_columns])
X = {name: f.to_numpy(dtype='float32') for name, f in X_frames.items()}

# 4. Target: log1p or raw, then standardized with fit-row statistics
def to_fit_scale(mfbtu):
    mfbtu = np.asarray(mfbtu, dtype=float)
    return np.log1p(mfbtu) if CONFIG['target'] == 'log1p' else mfbtu

t_fit = to_fit_scale(fit.MFBTU)
y_mean, y_std = t_fit.mean(), t_fit.std()

def encode_target(mfbtu):
    return ((to_fit_scale(mfbtu) - y_mean) / y_std).astype('float32')

def decode_to_kbtu(z):
    """Network output -> kBtu, clipped at 0 (team convention)."""
    t = np.asarray(z, dtype=float).ravel() * y_std + y_mean
    return np.maximum(np.expm1(t) if CONFIG['target'] == 'log1p' else t, 0)

y = {name: encode_target(f.MFBTU) for name, f in frames.items()}
y_kbtu = {name: f.MFBTU.to_numpy(dtype=float) for name, f in frames.items()}

# 5. Checks
assert len(fit) + len(monitor) == len(train) and set(fit.PUBID).isdisjoint(monitor.PUBID)
assert not set(ID_COLUMNS) & set(feature_columns)
assert all(np.isfinite(a).all() for a in X.values()) and all(np.isfinite(a).all() for a in y.values())
assert np.allclose(X_frames['fit'][continuous_columns].mean(), 0, atol=1e-8)
assert np.allclose(X_frames['fit'][continuous_columns].std(ddof=0), 1, atol=1e-8)
assert np.allclose(decode_to_kbtu(y['fit']), y_kbtu['fit'], rtol=1e-4, atol=10), 'target round trip failed'

display(pd.DataFrame({'Rows': [len(f) for f in frames.values()], 'Columns': [X[n].shape[1] for n in X]},
                     index=['fit (trains weights)', 'monitor (early stopping)', 'validation (compares versions)']))
print(f"{CONFIG['version']}: {len(feature_columns)} inputs "
      f"({len(continuous_columns)} scaled continuous, {len(binary_columns)} binary 0/1)")
print(f"Target '{CONFIG['target']}' standardized with fit-row mean {y_mean:.4f}, std {y_std:.4f}")
print('Stage 2 checks passed.')

,Rows,Columns
fit (trains weights),3781,175
monitor (early stopping),668,175
validation (compares versions),954,175


v03_raw_target: 175 inputs (10 scaled continuous, 165 binary 0/1)
Target 'raw' standardized with fit-row mean 15792948.4816, std 37780596.7062
Stage 2 checks passed.


## Cell 8: Build, train, and evaluate the network

The network is fully connected: inputs → Dense(64, ReLU) → Dense(32, ReLU) →
Dense(1, linear). It minimizes mean squared error on the standardized target (log1p or raw, as set in
`CONFIG`) with the Adam optimizer. Training stops when the monitor-set loss has not improved
for `patience` epochs, and the weights from the best epoch are restored.

Each version is trained with three seeds. We report validation MAE, RMSE, and R²
in kBtu as the mean ± standard deviation across seeds, so a later version only
counts as better if its gain is larger than this run-to-run variation.

In [6]:
# Cell 9: Model builder, metrics, and training function
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def build_model(n_inputs, cfg):
    reg = tf.keras.regularizers.l2(cfg['l2']) if cfg['l2'] > 0 else None
    model = tf.keras.Sequential([tf.keras.Input(shape=(n_inputs,))])
    for units in cfg['hidden_units']:
        model.add(tf.keras.layers.Dense(units, activation=cfg['activation'], kernel_regularizer=reg))
        if cfg['dropout'] > 0:
            model.add(tf.keras.layers.Dropout(cfg['dropout']))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['learning_rate']), loss='mse')
    return model

def kbtu_metrics(actual, predicted):
    return {'MAE_kBtu': mean_absolute_error(actual, predicted),
            'RMSE_kBtu': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)}

def train_one_seed(seed, cfg):
    tf.keras.utils.set_random_seed(seed)
    model = build_model(X['fit'].shape[1], cfg)
    stopper = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=cfg['patience'],
                                               restore_best_weights=True)
    # Keras labels validation_data as 'val_loss'; here it is our MONITOR set, not the official validation set
    history = model.fit(X['fit'], y['fit'], validation_data=(X['monitor'], y['monitor']),
                        epochs=cfg['max_epochs'], batch_size=cfg['batch_size'],
                        shuffle=True, verbose=0, callbacks=[stopper])
    preds = {name: decode_to_kbtu(model.predict(X[name], verbose=0)) for name in X}
    return model, history.history, preds

build_model(X['fit'].shape[1], CONFIG).summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │        11,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,377 (52.25 KB)

 Trainable params: 13,377 (52.25 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Cell 10: Train with each seed and evaluate in kBtu
import time

runs, rows = {}, []
max_train_kbtu = train.MFBTU.max()
for seed in CONFIG['seeds']:
    start = time.time()
    model, hist, preds = train_one_seed(seed, CONFIG)
    best_epoch = int(np.argmin(hist['val_loss'])) + 1
    runs[seed] = {'model': model, 'history': hist, 'preds': preds, 'best_epoch': best_epoch}
    for name in ['fit', 'monitor', 'validation']:
        assert np.isfinite(preds[name]).all(), f'Non-finite predictions ({name}, seed {seed})'
        rows.append({'seed': seed, 'partition': name, **kbtu_metrics(y_kbtu[name], preds[name]),
                     'max_pred_kBtu': preds[name].max(),
                     'n_above_train_max': int((preds[name] > max_train_kbtu).sum())})
    print(f'Seed {seed}: ran {len(hist["loss"])} epochs, best epoch {best_epoch}, {time.time() - start:.0f}s')

seed_results = pd.DataFrame(rows)
summary = seed_results.groupby('partition', sort=False)[['MAE_kBtu', 'RMSE_kBtu', 'R2']].agg(['mean', 'std'])
display(seed_results.round(3))
display(summary.round(3))

Seed 42: ran 51 epochs, best epoch 31, 23s
Seed 43: ran 31 epochs, best epoch 11, 14s
Seed 44: ran 34 epochs, best epoch 14, 14s


,seed,partition,MAE_kBtu,RMSE_kBtu,R2,max_pred_kBtu,n_above_train_max
0,42,fit,3210808.720,6.896887e+06,0.967,8.067061e+08,1
1,42,monitor,6652221.990,1.606434e+07,0.845,5.041134e+08,0
2,42,validation,5837744.646,1.518810e+07,0.827,4.202161e+08,0
3,43,fit,4418090.154,1.103875e+07,0.915,5.263748e+08,0
4,43,monitor,6257240.868,1.877093e+07,0.789,3.626529e+08,0
5,43,validation,5774990.143,1.430418e+07,0.846,3.413639e+08,0
6,44,fit,5053079.210,9.796770e+06,0.933,6.584351e+08,0
7,44,monitor,7814601.509,1.967818e+07,0.768,3.774240e+08,0
8,44,validation,7140530.515,1.573504e+07,0.814,3.704039e+08,0


MAE_kBtu                 RMSE_kBtu                  R2       
                   mean         std          mean          std   mean    std
partition                                                                   
fit         4227326.028  935832.973  9.244136e+06  2125515.051  0.938  0.026
monitor     6908021.456  809578.944  1.817115e+07  1880096.336  0.801  0.040
validation  6251088.435  770918.247  1.507577e+07   722011.946  0.829  0.016

In [8]:
# Cell 11: Record this version in the shared experiment log
import json
from datetime import datetime

LOG_PATH = data_path.parent / 'nn_experiment_log.csv'   # same Drive folder as the data
entry = {'version': CONFIG['version'], 'timestamp': datetime.now().isoformat(timespec='seconds'),
         'n_inputs': len(feature_columns),
         'mean_best_epoch': float(np.mean([r['best_epoch'] for r in runs.values()]))}
for part, prefix in [('validation', 'val'), ('fit', 'fit')]:
    part_rows = seed_results[seed_results.partition.eq(part)]
    for metric in ['MAE_kBtu', 'RMSE_kBtu', 'R2']:
        entry[f'{prefix}_{metric}_mean'] = part_rows[metric].mean()
        entry[f'{prefix}_{metric}_std'] = part_rows[metric].std()
entry['config'] = json.dumps(CONFIG)

log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
if not log.empty:
    log = log[log['version'].ne(CONFIG['version'])]   # rerunning a version replaces its old row
log = pd.concat([log, pd.DataFrame([entry])], ignore_index=True)
log.to_csv(LOG_PATH, index=False)
display(log.drop(columns='config').round(3))
print(f'Logged {CONFIG["version"]} to {LOG_PATH}')

,version,timestamp,n_inputs,mean_best_epoch,val_MAE_kBtu_mean,val_MAE_kBtu_std,val_RMSE_kBtu_mean,val_RMSE_kBtu_std,val_R2_mean,val_R2_std,fit_MAE_kBtu_mean,fit_MAE_kBtu_std,fit_RMSE_kBtu_mean,fit_RMSE_kBtu_std,fit_R2_mean,fit_R2_std
0,v01_base_log,2026-11-14T23:53:01,175,8.000,5237602.429,181773.519,1.654535e+07,1852171.543,0.793,0.046,4937323.867,97952.862,1.814874e+07,608745.647,0.769,0.015
1,v02_engineered_log,2026-11-15T00:04:09,183,5.000,5207122.898,396135.435,1.703522e+07,402956.017,0.782,0.010,5177520.145,540436.732,1.972582e+07,3405880.332,0.722,0.098
2,v03_raw_target,2026-11-15T00:15:21,175,18.667,6251088.435,770918.247,1.507577e+07,722011.946,0.829,0.016,4227326.028,935832.973,9.244136e+06,2125515.051,0.938,0.026


Logged v03_raw_target to /content/drive/MyDrive/Data/nn_experiment_log.csv


## Cell 12: v03 results

Training on the standardized raw target gave validation MAE 6.25M ± 0.77M kBtu,
RMSE 15.1M ± 0.7M kBtu, and R² 0.829 ± 0.016 (3 seeds). Compared with v01 (log1p),
MAE worsened by 1.0M kBtu, well beyond seed variation, while RMSE and R² improved.
The same trade-off appeared in the gradient boosting experiments, indicating it is
driven by the target transform rather than the algorithm: a raw-scale squared-error
loss is dominated by the largest buildings, improving fit to them at the expense of
small and mid-sized buildings.

The raw target also widened the generalization gap (fit R² 0.938 vs monitor 0.801)
and increased seed variability. Because validation MAE is the team's selection
metric, log1p remains the target for subsequent versions.